# YOLO Training

Instructions:

*   A directory of images and corresponding 'annotation.json' file should already exist before running this. See 'plume_bbox_annotator.ipynb'

*   Change runtime type to 'T4' (top right corner)

*   As with the bounding boxes, update paths in block below

*   In the final block update the 'epochs=' number. The higher this number, the longer the training will take but performance should be better. If experimenting with the code, leave at 10, once you're confident with the training data, increase this to 100.

* After finishes running it should generate some files in a /runs/ folder. A new folder will be created everytime you run the train function. This will contain the final model weights 'best.pt' and some statistics on the accuracy of the model on validation set of images.

* If you're runtime is disconnected before it finsihed training, you can reload the partially trained model as it saves a .pt file every epoch. Change 'model = YOLO()' to model = YOLO('PATH_TO_PT_FILE')

In [2]:
# mount drive and set dir
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Directory structure overview:
# HOME/
# ├── images/              # Contains original input images (already exists)
# ├── annotations.json     # JSON file with metadata or labels for the images (already exists)
# └── output/              # Everything bellow will be created to store processed results
#     ├── images/          # Holds transformed or annotated versions of input images
#         ├── train/
#         ├── val/
#         └── test/
#     └── labels/          # Stores label files (e.g., YOLO format) generated from annotations
#         ├── train/
#         ├── val/
#         └── test/

HOME = "/content/drive/MyDrive/Colab Notebooks/thermal_training"
os.chdir(HOME)
print("HOME:", HOME)

HOME: /content/drive/MyDrive/Colab Notebooks/thermal_training


In [4]:
annotations_path = os.path.join(HOME, 'annotations.json')
assert os.path.exists(annotations_path), f"Annotations file not found at {annotations_path}"
images_dir = os.path.join(HOME, 'images')
assert os.path.exists(images_dir), f"Images directory not found at {images_dir}"
# images_dir = "/content/drive/MyDrive/Colab Notebooks/test/training_images"
output_dir = os.path.join(HOME, 'output')
if not os.path.exists(output_dir):
  os.makedirs(output_dir)
labels_dir = os.path.join(HOME, 'labels')
if not os.path.exists(labels_dir):
  os.makedirs(labels_dir)

In [5]:
# TODO - this could be infered fromthe annotations file to reduce manual duplication from other file
class_dict = {'plume':0}
class_list = ['plume']

## Convert bboxes to correct format

In [6]:
import os
import shutil
import random
import re

def split_data(images_dir, labels_dir, output_dir, val_split=0.2, test_split=0.1):
    """Splits the images and labels into train, validation, and test sets and copies them to a new directory.

    Args:
        images_dir (str): Path to the directory containing images.
        labels_dir (str): Path to the directory containing labels.
        output_dir (str): Path to the output directory where the split data will be stored.
        val_split (float): Fraction of data to use for validation.
        test_split (float): Fraction of data to use for testing.

    Returns:
        None
    """

    # Clean/delete the output directory if it exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)  # Remove the directory and its contents
        print(f"Cleaned output directory: {output_dir}")

    # Create train, val, and test directories within the output directory
    os.makedirs(os.path.join(output_dir, "images", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "images", "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "images", "test"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "train"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "val"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels", "test"), exist_ok=True)

    # Get list of image files
    image_files = os.listdir(images_dir)
    image_files = [f for f in image_files if f.endswith(('.png', '.jpg', '.jpeg'))]  # Filter for image files
    print(f"image_files: {image_files}")

    # # Rename image files and update the list
    # renamed_image_files = []
    # for image_file in image_files:
    #     new_name = re.sub(r"\.\d+", "", image_file)  # Remove ".number" using regex
    #     os.rename(os.path.join(images_dir, image_file), os.path.join(images_dir, new_name))
    #     renamed_image_files.append(new_name)
    # image_files = renamed_image_files  # Update the list with renamed files

        # Filter out images without corresponding labels
    labeled_images = []
    for image in image_files:
        # print(f"image: {image}")
        label = image.replace(os.path.splitext(image)[1], ".txt")  # Assuming labels have the same name as images but with .txt extension
        label_path = os.path.join(labels_dir, label)
        # print(f"looking for label: {label_path}")
        if os.path.exists(label_path):
            labeled_images.append(image)
        else:
            print(f"Warning: Ignoring unlabeled image: {image}")

    # Shuffle the image files randomly
    random.shuffle(labeled_images)

    # Calculate split indices
    num_images = len(labeled_images)
    num_val = int(val_split * num_images)
    num_test = int(test_split * num_images)
    num_train = num_images - num_val - num_test

    # Split the image files into train, val, and test sets
    train_images = labeled_images[:num_train]
    print(f"train_images: {train_images}")
    val_images = labeled_images[num_train:num_train + num_val]
    test_images = labeled_images[num_train + num_val:]


    # Copy images and labels to their respective directories
    for image in train_images:
        shutil.copy(os.path.join(images_dir, image), os.path.join(output_dir, "images", "train", image))
        label = image.replace(os.path.splitext(image)[1], ".txt")  # Assuming labels have the same name as images but with .txt extension
        # print(f"label: {label}")
        # print(f"os.path.join(labels_dir, label): {os.path.join(labels_dir, label)}")
        # print(f"os.path.join(output_dir, 'labels', 'train', label): {os.path.join(output_dir, 'labels', 'train', label)}")
        shutil.copy(os.path.join(labels_dir, label), os.path.join(output_dir, "labels", "train", label))

    for image in val_images:
        shutil.copy(os.path.join(images_dir, image), os.path.join(output_dir, "images", "val", image))
        label = image.replace(os.path.splitext(image)[1], ".txt")
        shutil.copy(os.path.join(labels_dir, label), os.path.join(output_dir, "labels", "val", label))

    for image in test_images:
        shutil.copy(os.path.join(images_dir, image), os.path.join(output_dir, "images", "test", image))
        label = image.replace(os.path.splitext(image)[1], ".txt")
        shutil.copy(os.path.join(labels_dir, label), os.path.join(output_dir, "labels", "test", label))



In [7]:
# helper function to creat yolo yaml file
import yaml

def generate_yaml(path, train, val, test, names, output_dir, yaml_filename="fuego.yaml"):
    """Generates a YAML file for YOLOv8 segmentation model training.

    Args:
        path (str): Path to the dataset root directory.
        train (str): Path to the training images (relative to 'path').
        val (str): Path to the validation images (relative to 'path').
        test (str, optional): Path to the test images (relative to 'path'). Defaults to None.
        names (list): List of class names.

    Returns:
        None
    """

    data = {
        "path": path,
        "train": train,
        "val": val,
        "test": test,
        "names": {i: name for i, name in enumerate(names)},
    }

    with open(os.path.join(output_dir,yaml_filename), "w") as f:
        yaml.dump(data, f, indent=4)
    print(f"YAML file generated at: {os.path.join(output_dir,yaml_filename)}")

In [8]:
# chage bounding box labels ointo the format yolo expects
import json
import cv2

with open(annotations_path, 'r') as f:
  annotations = json.load(f)

# class_dict = {'plume':0, 'summit':1}

for f,b in annotations.items():
  # get an example image to get size
  img = cv2.imread(os.path.join(images_dir, f))
  img_width = img.shape[1]
  img_height = img.shape[0]
  # img_width = 3840
  # img_height = 2160

  # class x_center y_center width height
  label_text=""
  for bbox in annotations[f]:
    # TODO - temporarily change x,y to be centre of bbox but should do this in tool
    normalised_bbox = {
        # 'x': (bbox['x'] + bbox['width']/2)/ img_width,
        # 'y': (bbox['y'] + bbox['height']/2) / img_height,
        'x': (bbox['x'] + bbox['width']/2)/ img_width,
        'y': (bbox['y'] + bbox['height']/2) / img_height,
        'width': bbox['width'] / img_width,
        'height': bbox['height'] / img_height
    }
    label_text += f"{class_dict[bbox['label']]} {normalised_bbox['x']} {normalised_bbox['y']} {normalised_bbox['width']} {normalised_bbox['height']}\n"

  label_path = os.path.join(labels_dir, f"{f.split('.')[0]}.txt")
  with open(label_path, 'w+') as text_file:
      # print(f"writing file: {label_path}")
      text_file.write(label_text)

In [10]:
# split into test, training and validation sets
split_data(images_dir, labels_dir, output_dir, val_split=0.2, test_split=0.1)


Cleaned output directory: /content/drive/MyDrive/Colab Notebooks/thermal_training/output
image_files: ['frame_87.png', 'frame_143.png', 'frame_231.png', 'frame_191.png', 'frame_71.png', 'frame_187.png', 'frame_175.png', 'frame_15.png', 'frame_67.png', 'frame_75.png', 'frame_127.png', 'frame_79.png', 'frame_139.png', 'frame_95.png', 'frame_171.png', 'frame_227.png', 'frame_219.png', 'frame_115.png', 'frame_183.png', 'frame_27.png', 'frame_135.png', 'frame_235.png', 'frame_7.png', 'frame_23.png', 'frame_199.png', 'frame_131.png', 'frame_83.png', 'frame_207.png', 'frame_63.png', 'frame_47.png', 'frame_39.png', 'frame_55.png', 'frame_223.png', 'frame_31.png', 'frame_107.png', 'frame_167.png', 'frame_111.png', 'frame_159.png', 'frame_211.png', 'frame_151.png', 'frame_215.png', 'frame_163.png', 'frame_59.png', 'frame_239.png', 'frame_51.png', 'frame_3.png', 'frame_91.png', 'frame_19.png', 'frame_123.png', 'frame_103.png', 'frame_203.png', 'frame_43.png', 'frame_155.png', 'frame_147.png', 'fr

In [11]:
# generate yaml for yolo training
path = output_dir
train = "images/train"
val = "images/val"
test = "images/test"
names = class_list
generate_yaml(path, train, val, test, names, output_dir, yaml_filename="fuego.yaml")
yaml_path = os.path.join(output_dir, "fuego.yaml")


YAML file generated at: /content/drive/MyDrive/Colab Notebooks/thermal_training/output/fuego.yaml


## Train YOLO

In [ ]:
# install yolo
!pip install ultralytics -q

In [12]:
from ultralytics import YOLO
import os

print(os.getcwd())

# Load a model
# model = YOLO("/home/mark/Work/PhD/volcanic_vision/runs/segment/train7/weights/last.pt")  # load a pretrained model (recommended for training)
model = YOLO()

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {device}')
model = model.to(device)

# Train the model
# yaml_path = "E:\\data\\plume_data\\santiaguito_yolo_training\\test4\\santiaguito.yaml"
save_dir = os.path.join(HOME, 'runs')
results = model.train(data=yaml_path, epochs=100, imgsz=720, pretrained=False, plots=True, save=True, save_dir=save_dir, workers=0)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/content/drive/MyDrive/Colab Notebooks/thermal_training
Using device: cuda
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/thermal_training/output/fuego.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, i